# Construction PPE Detection — Train on Kaggle (offline-capable)

Trains YOLOv8 on the **PPE dataset** (`segp-fcn6m/ppe-yezzu-fwbjo`, ~33k train images, 5 classes: `Helmet`, `No-Helmet`, `No-Vest`, `Person`, `Vest`) and evaluates on the held-out **test split (4,190 images)** with per-class precision / recall / F1 / mAP — flagging which classes clear **90%**.

### Why Kaggle (vs free Colab)
- **Save & Run All (Commit)** runs the whole notebook **offline for up to 12 h** — close the browser, it finishes and saves the output. No session babysitting.
- **4 CPU cores** (vs Colab free's 2) → eases the data-loading bottleneck → faster.
- **T4 x2** or P100 16 GB, ~30 GPU-hours/week.

### Setup (right-hand **Settings** panel)
1. **Accelerator → GPU T4 x2** ← **recommended** (the **P100 is older "Pascal" hardware and is incompatible with current PyTorch builds** — it errors at training time). The T4 works out of the box.
2. **Internet → On** (required for the dataset download + pip).
3. *(optional)* **Add-ons → Secrets** → add `ROBOFLOW_API_KEY`.

> First time using an accelerator? Kaggle requires a one-time **phone verification** (your profile → Settings → Phone Verification) before the GPU options become selectable.

### Run it
- Test interactively first (run cells 1–6 to confirm the data downloads), **then** use **Save Version → Save & Run All (Commit)** to train offline. When it finishes, grab `best.pt` from the notebook's **Output** tab.

## 1. Check GPU + CPU

In [ ]:
import os, torch
print('CPUs:', os.cpu_count())
print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    print('NO GPU — Settings > Accelerator > GPU, then restart.')

## 2. Install dependencies (needs Internet = On)

In [ ]:
# NOTE: no `-U` — upgrading torch can pull a build that drops support for older
# GPUs (e.g. the P100 / Pascal sm_60). Install against Kaggle's existing torch.
!pip install -q ultralytics roboflow
import torch, ultralytics, roboflow
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '| roboflow', roboflow.__version__)

## 3. Roboflow API key

**Recommended:** **Add-ons → Secrets**, add a secret `ROBOFLOW_API_KEY`, attach it to the notebook. Otherwise paste it in the fallback line (and rotate it later on Roboflow).

In [ ]:
import os
ROBOFLOW_API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    pass
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
assert ROBOFLOW_API_KEY, (
    'No Roboflow API key found. Add a Kaggle Secret named ROBOFLOW_API_KEY '
    '(Add-ons -> Secrets, then attach it to this notebook), or set the '
    'ROBOFLOW_API_KEY environment variable. Get your key at '
    'https://app.roboflow.com (Settings -> API).'
)

## 4. Download the dataset (~2.8 GB) to /kaggle/working

In [ ]:
import shutil, os
DEST = '/kaggle/working/ppe'
shutil.rmtree(DEST, ignore_errors=True)   # roboflow SKIPS download if the folder exists — wipe for a clean fetch

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('segp-fcn6m').project('ppe-yezzu-fwbjo')
dataset = project.version(1).download('yolov8', location=DEST)
DATA_DIR = dataset.location
print('Downloaded to:', DATA_DIR)
print('contains:', os.listdir(DATA_DIR))   # expect train, valid, test, data.yaml

## 5. Fix `data.yaml` paths

Roboflow's export uses relative paths that don't resolve, and may nest the data in a subfolder. Locate the real `data.yaml` and rewrite with absolute paths.

In [ ]:
import glob, yaml
matches = glob.glob('/kaggle/working/**/data.yaml', recursive=True)
assert matches, 'No data.yaml found — re-run the download cell (Cell 4).'
yml = matches[0]
DATA_DIR = os.path.dirname(yml)
spec = yaml.safe_load(open(yml))
spec['path'] = DATA_DIR
spec['train'] = 'train/images'
spec['val'] = 'valid/images'
spec['test'] = 'test/images'
yaml.safe_dump(spec, open(yml, 'w'), sort_keys=False)
print('data.yaml:', yml)
print('DATA_DIR :', DATA_DIR, '| classes:', spec['names'])
for k in ('train', 'val', 'test'):
    p = os.path.join(DATA_DIR, spec[k])
    print(f'  {k}: {p} -> exists: {os.path.isdir(p)}')

## 6. Train

Outputs go to `/kaggle/working/runs`, which Kaggle **persists as the notebook's output** on commit — no Drive needed. With 4 CPU cores feeding the GPU, this should finish 30 epochs well within the 12 h commit limit.

- **batch=48** uses the GPU's spare memory (~11 GB; auto-batch tends to under-pick). Bump to 64 if it fits, lower to 32 on OOM.
- **T4 x2 users:** set `DEVICE = '0,1'` below to use both GPUs (validation still runs on one).

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolov8s.pt'   # 'yolov8m.pt' for more capacity (this much data can use it)
EPOCHS = 30
IMGSZ = 640
BATCH = 48                  # try 64 if no OOM; 32 if it OOMs
DEVICE = 0                  # use '0,1' on a T4 x2 runtime for multi-GPU
PROJECT_DIR = '/kaggle/working/runs'
RUN_NAME = 'ppe_s'

model = YOLO(BASE_MODEL)
model.train(
    data=yml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=12,
    device=DEVICE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)
BEST = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
print('Best checkpoint:', BEST)

## 7. Evaluate on the held-out test split (4,190 images)

Authoritative per-class precision / recall / F1 / mAP@50 / mAP@50-95, with a 90% check per class.

In [ ]:
def evaluate(weights, tta=False):
    m = YOLO(weights)
    res = m.val(data=yml, split='test', imgsz=IMGSZ, augment=tta, device=0, verbose=False)
    box = res.box
    names = res.names
    idx = list(box.ap_class_index)
    p, r, f1 = list(box.p), list(box.r), list(box.f1)
    ap50, ap = list(box.ap50), list(box.ap)
    print('=' * 78)
    print(f" Per-class metrics on TEST{'  (TTA)' if tta else ''}")
    print('=' * 78)
    print(f"{'class':<12}{'precision':>11}{'recall':>10}{'F1':>9}{'mAP@50':>10}{'mAP50-95':>11}{'  90%?':>8}")
    n90 = 0
    for j, ci in enumerate(idx):
        name = names[int(ci)]
        row = dict(precision=float(p[j]), recall=float(r[j]), f1=float(f1[j]),
                   map50=float(ap50[j]), map5095=float(ap[j]))
        hit = all(row[k] >= 0.90 for k in ('precision', 'recall', 'f1', 'map50'))
        n90 += hit
        f = lambda x: f'{x*100:6.1f}%'
        print(f"{name:<12}{f(row['precision']):>11}{f(row['recall']):>10}{f(row['f1']):>9}"
              f"{f(row['map50']):>10}{f(row['map5095']):>11}{('  YES' if hit else ''):>8}")
    mp, mr = float(box.mp), float(box.mr)
    mf1 = 2*mp*mr/(mp+mr) if (mp+mr) else 0.0
    print('-' * 78)
    print(f"{'ALL (mean)':<12}{mp*100:10.1f}%{mr*100:9.1f}%{mf1*100:8.1f}%"
          f"{float(box.map50)*100:9.1f}%{float(box.map)*100:10.1f}%")
    print(f"\nClasses clearing 90% on P/R/F1/mAP@50: {n90}/{len(idx)}")
    return res

_ = evaluate(BEST)

## 8. (Optional) Test-time augmentation — usually +1-3 mAP for free

In [ ]:
_ = evaluate(BEST, tta=True)

## 9. Save the model for download

Copies `best.pt` to `/kaggle/working/` so it's easy to grab from the notebook's **Output** tab after the commit finishes.

In [ ]:
shutil.copy(BEST, '/kaggle/working/best.pt')
print('Saved /kaggle/working/best.pt')
# Locally: put it in your project as models/ppe_s.pt, then
#   python eval_ppe.py --model models/ppe_s.pt --dataset-dir data/ppe_download